# Whisper 语音识别教程

本教程介绍 OpenAI Whisper 风格的语音识别模型，包括：

1. **模型架构** - 编码器-解码器 Transformer
2. **音频编码器** - 卷积下采样 + Transformer
3. **文本解码器** - 自回归 Transformer
4. **多任务能力** - 转录、翻译、语言识别
5. **训练与推理** - 损失函数与生成策略

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

# 添加 src 目录到路径
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

# 检查设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Whisper 模型架构

Whisper 采用标准的编码器-解码器 Transformer 架构：

```
┌─────────────────────────────────────────────────────────────┐
│                        Whisper                               │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  音频 ──→ [Log-Mel] ──→ [卷积下采样] ──→ [Transformer 编码器] │
│                                              ↓               │
│                                         音频特征             │
│                                              ↓               │
│  <|sot|> ──→ [Token 嵌入] ──→ [Transformer 解码器] ──→ 文本  │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
from whisper import WhisperConfig, Whisper, create_whisper_model

# 查看默认配置
config = WhisperConfig()
print("Whisper 默认配置:")
print(f"  n_mels: {config.n_mels}")
print(f"  d_model: {config.d_model}")
print(f"  n_heads: {config.n_heads}")
print(f"  n_encoder_layers: {config.n_encoder_layers}")
print(f"  n_decoder_layers: {config.n_decoder_layers}")
print(f"  vocab_size: {config.vocab_size}")

In [ ]:
# 创建不同大小的模型
print("不同大小的 Whisper 模型:")
for size in ["tiny", "base", "small"]:
    model = create_whisper_model(size)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {size}: d_model={model.config.d_model}, params={n_params/1e6:.1f}M")

## 2. 音频编码器

音频编码器将 Log-Mel 频谱图转换为音频特征序列：

1. **卷积下采样**: 两层 1D 卷积，stride=2，将时间维度减半
2. **位置编码**: 正弦位置编码
3. **Transformer 编码器**: 多层自注意力

In [ ]:
from whisper import AudioEncoder, SinusoidalPositionalEncoding

# 创建音频编码器
encoder_config = WhisperConfig(
    n_mels=80,
    d_model=256,
    n_heads=4,
    n_encoder_layers=4,
    d_ff=1024
)
encoder = AudioEncoder(encoder_config)

# 模拟输入: [batch, n_mels, time]
mel_input = torch.randn(2, 80, 100)
encoder_output = encoder(mel_input)

print(f"输入 Mel 频谱形状: {mel_input.shape}")
print(f"编码器输出形状: {encoder_output.shape}")
print(f"  - 时间维度从 {mel_input.shape[2]} 下采样到 {encoder_output.shape[1]}")

In [ ]:
# 可视化位置编码
pe = SinusoidalPositionalEncoding(d_model=256)
x = torch.zeros(1, 100, 256)
pe_output = pe(x)

# 提取位置编码 (输出 - 输入)
pos_encoding = pe_output[0].detach().numpy()

plt.figure(figsize=(12, 5))
plt.imshow(pos_encoding.T, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Sinusoidal Positional Encoding')
plt.tight_layout()
plt.show()

## 3. 多头注意力机制

Transformer 的核心是多头注意力：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
from whisper import MultiHeadAttention

# 创建多头注意力
mha = MultiHeadAttention(d_model=256, n_heads=4)

# 自注意力
x = torch.randn(2, 50, 256)
self_attn_output = mha(x, x, x)
print(f"自注意力输出形状: {self_attn_output.shape}")

# 交叉注意力
query = torch.randn(2, 30, 256)
key_value = torch.randn(2, 50, 256)
cross_attn_output = mha(query, key_value, key_value)
print(f"交叉注意力输出形状: {cross_attn_output.shape}")

In [ ]:
# 可视化因果注意力掩码
seq_len = 10
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

plt.figure(figsize=(6, 5))
plt.imshow(~causal_mask, cmap='Blues')
plt.colorbar()
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Causal Attention Mask\n(White = Attend, Blue = Mask)')
plt.tight_layout()
plt.show()

print("因果掩码确保每个位置只能看到之前的位置")

## 4. 文本解码器

文本解码器自回归地生成文本 token：

1. **Token 嵌入**: 将 token ID 转换为向量
2. **位置编码**: 添加位置信息
3. **Transformer 解码器**: 自注意力 + 交叉注意力

In [ ]:
from whisper import TextDecoder

# 创建文本解码器
decoder_config = WhisperConfig(
    vocab_size=1000,
    d_model=256,
    n_heads=4,
    n_decoder_layers=4,
    d_ff=1024
)
decoder = TextDecoder(decoder_config)

# 模拟输入
tokens = torch.randint(0, 1000, (2, 20))  # [batch, seq_len]
encoder_output = torch.randn(2, 50, 256)  # 编码器输出

decoder_output = decoder(tokens, encoder_output)

print(f"输入 token 形状: {tokens.shape}")
print(f"编码器输出形状: {encoder_output.shape}")
print(f"解码器输出形状: {decoder_output.shape}")

## 5. 完整模型前向传播

In [ ]:
# 创建完整的 Whisper 模型
model_config = WhisperConfig(
    n_mels=80,
    vocab_size=1000,
    d_model=256,
    n_heads=4,
    n_encoder_layers=4,
    n_decoder_layers=4,
    d_ff=1024
)
model = Whisper(model_config)

# 模拟输入
mel = torch.randn(2, 80, 100)  # [batch, n_mels, time]
tokens = torch.randint(0, 1000, (2, 20))  # [batch, seq_len]

# 前向传播
logits = model(mel, tokens)

print(f"Mel 输入形状: {mel.shape}")
print(f"Token 输入形状: {tokens.shape}")
print(f"输出 logits 形状: {logits.shape}")
print(f"  - 每个位置预测 {logits.shape[-1]} 个词汇的概率")

In [ ]:
# 分步执行: 编码和解码
# 1. 编码音频
encoder_output = model.encode(mel)
print(f"编码器输出形状: {encoder_output.shape}")

# 2. 解码文本
logits = model.decode(tokens, encoder_output)
print(f"解码器输出形状: {logits.shape}")

## 6. 多任务能力

Whisper 通过特殊 token 支持多种任务：

```
<|startoftranscript|>  开始转录
<|en|>                 语言标识 (英语)
<|zh|>                 语言标识 (中文)
<|transcribe|>         转录任务
<|translate|>          翻译任务 (翻译为英语)
<|notimestamps|>       不输出时间戳
<|endoftext|>          结束
```

In [ ]:
# 模拟特殊 token
SPECIAL_TOKENS = {
    "<|startoftranscript|>": 0,
    "<|en|>": 1,
    "<|zh|>": 2,
    "<|transcribe|>": 3,
    "<|translate|>": 4,
    "<|notimestamps|>": 5,
    "<|endoftext|>": 6,
}

# 转录中文任务的 token 序列
transcribe_zh = [
    SPECIAL_TOKENS["<|startoftranscript|>"],
    SPECIAL_TOKENS["<|zh|>"],
    SPECIAL_TOKENS["<|transcribe|>"],
    SPECIAL_TOKENS["<|notimestamps|>"],
    # ... 后续是生成的文本 token
]

# 翻译为英语任务的 token 序列
translate_to_en = [
    SPECIAL_TOKENS["<|startoftranscript|>"],
    SPECIAL_TOKENS["<|zh|>"],
    SPECIAL_TOKENS["<|translate|>"],
    SPECIAL_TOKENS["<|notimestamps|>"],
    # ... 后续是生成的英文 token
]

print("转录中文任务 token:", transcribe_zh)
print("翻译为英语任务 token:", translate_to_en)

## 7. 训练损失函数

Whisper 使用标准的交叉熵损失：

$$L = -\sum_{t=1}^{T} \log P(y_t | y_{<t}, x)$$

In [ ]:
def whisper_loss(logits, targets, ignore_index=-100):
    """
    计算 Whisper 损失
    
    Args:
        logits: 模型输出 [batch, seq_len, vocab_size]
        targets: 目标 token [batch, seq_len]
        ignore_index: 忽略的 token (如 padding)
    """
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1),
        ignore_index=ignore_index
    )
    return loss

# 计算损失
targets = torch.randint(0, 1000, (2, 20))
loss = whisper_loss(logits, targets)
print(f"损失值: {loss.item():.4f}")

## 8. 自回归生成

推理时，模型自回归地生成 token：

In [ ]:
@torch.no_grad()
def greedy_decode(model, mel, max_len=50, start_token=0, end_token=6):
    """
    贪婪解码
    
    Args:
        model: Whisper 模型
        mel: Mel 频谱 [batch, n_mels, time]
        max_len: 最大生成长度
        start_token: 起始 token
        end_token: 结束 token
    """
    batch_size = mel.size(0)
    device = mel.device
    
    # 编码音频
    encoder_output = model.encode(mel)
    
    # 初始化 token 序列
    tokens = torch.full((batch_size, 1), start_token, dtype=torch.long, device=device)
    
    for _ in range(max_len):
        # 解码
        logits = model.decode(tokens, encoder_output)
        
        # 取最后一个位置的预测
        next_token_logits = logits[:, -1, :]
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)
        
        # 添加到序列
        tokens = torch.cat([tokens, next_token], dim=1)
        
        # 检查是否所有序列都生成了结束 token
        if (next_token == end_token).all():
            break
    
    return tokens

# 测试贪婪解码
model.eval()
generated = greedy_decode(model, mel, max_len=30)
print(f"生成的 token 序列形状: {generated.shape}")
print(f"生成的 token: {generated[0].tolist()[:15]}...")

## 9. 模型参数统计

In [ ]:
def count_parameters(model):
    """统计模型参数"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def model_summary(model):
    """打印模型摘要"""
    print("模型组件参数统计:")
    for name, module in model.named_children():
        params = sum(p.numel() for p in module.parameters())
        print(f"  {name}: {params/1e6:.2f}M")
    
    total, trainable = count_parameters(model)
    print(f"\n总参数: {total/1e6:.2f}M")
    print(f"可训练参数: {trainable/1e6:.2f}M")

model_summary(model)

## 10. 总结

本教程介绍了 Whisper 语音识别模型的核心组件：

| 组件 | 功能 | 关键特性 |
|:-----|:-----|:---------|
| 音频编码器 | 提取音频特征 | 卷积下采样 + Transformer |
| 文本解码器 | 生成文本 | 自回归 + 交叉注意力 |
| 多任务 | 转录/翻译 | 特殊 token 控制 |

**Whisper 的优势**:
- 多语言支持
- 多任务能力
- 强鲁棒性

**下一步**: 学习 TTS 文本转语音模型